# Chess Autoresearch - Experiment Analysis

Load `results.tsv` and visualize ELO progression across experiments.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv('results.tsv', sep='\t')
print(f"Total experiments: {len(df)}")
print(f"Status counts:\n{df['status'].value_counts().to_string()}")
print(f"\nKept experiments:")
kept = df[df['status'] == 'keep']
print(kept[['commit', 'estimated_elo', 'description']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Filter out crashes for plotting
valid = df[df['status'] != 'crash'].copy()
valid['experiment'] = range(len(valid))

# Plot all experiments
colors = valid['status'].map({'keep': '#4ecca3', 'discard': '#888888'}).fillna('#888888')
ax.scatter(valid['experiment'], valid['estimated_elo'], c=colors, s=40, zorder=3, alpha=0.8)

# Running maximum (best ELO so far)
best_so_far = valid['estimated_elo'].cummax()
ax.step(valid['experiment'], best_so_far, where='post', color='#e94560', linewidth=2, label='Best ELO')

# Labels on kept experiments
for _, row in valid[valid['status'] == 'keep'].iterrows():
    ax.annotate(row['description'][:30], (row['experiment'], row['estimated_elo']),
                textcoords='offset points', xytext=(5, 8), fontsize=7, color='#4ecca3', rotation=15)

ax.set_xlabel('Experiment #')
ax.set_ylabel('Estimated ELO')
ax.set_title('Chess Autoresearch - ELO Progression')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.savefig('progress.png', dpi=150)
plt.show()

if len(kept) >= 2:
    baseline = kept.iloc[0]['estimated_elo']
    best = kept.iloc[-1]['estimated_elo']
    print(f"\nBaseline ELO: {baseline}")
    print(f"Best ELO: {best}")
    print(f"Total improvement: +{best - baseline:.1f} ELO")